# Task 2 — Understanding ViT

ATML PA0 — Task 2

Sections:
1. Pretrained ViT classification
2. Visualizing patch attention (CLS token, final layer)
3. Analysis of attention map (written discussion)
4. Patch-masking experiment
5. CLS token vs mean-pooled patch tokens (linear probes)

## Setup

In [ ]:
# !pip install -q transformers torch torchvision matplotlib pillow scikit-learn

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from transformers import ViTImageProcessor, ViTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("results", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

## 1. Using a Pre-trained ViT for Image Classification

We use `google/vit-base-patch16-224`, pretrained on ImageNet-21k and fine-tuned on ImageNet-1k. Patch size 16, input 224x224 -> 14x14 = 196 patches + 1 CLS token.

In [ ]:
MODEL_NAME = "google/vit-base-patch16-224"

processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
model = ViTForImageClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Num labels:", model.config.num_labels)
print("Patch size:", model.config.patch_size)
print("Image size:", model.config.image_size)

In [ ]:
# TODO: pick 1-3 images. Easiest: download a few sample images by URL.
# Replace these with images of your choice (a dog, a cat, any object with a clear ImageNet-ish class).

import urllib.request

sample_urls = {
    "dog": "https://images.unsplash.com/photo-1552053831-71594a27632d?w=500",
    "cat": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=500",
}

images = {}
for name, url in sample_urls.items():
    path = f"results/{name}.jpg"
    urllib.request.urlretrieve(url, path)
    images[name] = Image.open(path).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4))
if len(images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
@torch.no_grad()
def classify(img):
    inputs = processor(images=img, return_tensors="pt").to(device)
    outputs = model(**inputs)
    logits = outputs.logits
    pred_idx = logits.argmax(-1).item()
    pred_label = model.config.id2label[pred_idx]
    confidence = torch.softmax(logits, dim=-1)[0, pred_idx].item()
    return pred_label, confidence

for name, img in images.items():
    label, conf = classify(img)
    print(f"{name}: predicted '{label}' (confidence {conf:.3f})")

**Discussion:** Record the top-1 prediction for each image and state whether it seems reasonable.

> TODO: your answer here.

## 2. Visualizing Patch Attention

In [ ]:
@torch.no_grad()
def get_attention_map(img, model, processor):
    inputs = processor(images=img, return_tensors="pt").to(device)
    outputs = model(**inputs, output_attentions=True)
    # outputs.attentions: tuple of (num_layers) tensors, each (batch, heads, seq_len, seq_len)
    last_layer_attn = outputs.attentions[-1]  # (1, num_heads, 197, 197)

    # Average over heads
    attn_avg = last_layer_attn.mean(dim=1)  # (1, 197, 197)

    # CLS token (index 0) attending to all patch tokens (indices 1:197)
    cls_attn = attn_avg[0, 0, 1:]  # (196,)

    # Reshape to 14x14 spatial grid
    num_patches_per_side = int(np.sqrt(cls_attn.shape[0]))  # 14
    attn_map = cls_attn.reshape(num_patches_per_side, num_patches_per_side).cpu().numpy()

    return attn_map, outputs


def overlay_attention(img, attn_map, alpha=0.5):
    from PIL import Image as PILImage
    img_resized = img.resize((224, 224))
    # Upsample attention map to 224x224 using PIL for simplicity
    attn_img = PILImage.fromarray((attn_map / attn_map.max() * 255).astype(np.uint8))
    attn_upsampled = attn_img.resize((224, 224), PILImage.BILINEAR)
    attn_upsampled = np.array(attn_upsampled) / 255.0
    return np.array(img_resized), attn_upsampled


fig, axes = plt.subplots(len(images), 2, figsize=(8, 4*len(images)))
if len(images) == 1:
    axes = axes.reshape(1, 2)

for i, (name, img) in enumerate(images.items()):
    attn_map, _ = get_attention_map(img, model, processor)
    img_arr, attn_upsampled = overlay_attention(img, attn_map)

    axes[i, 0].imshow(img_arr)
    axes[i, 0].set_title(f"{name} (original)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(img_arr)
    axes[i, 1].imshow(attn_upsampled, cmap="jet", alpha=0.5)
    axes[i, 1].set_title(f"{name} (attention overlay)")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("../figures/task2_attention_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Analyze the Attention Map

**Discussion prompts:**
- Did the ViT focus on regions corresponding to the predicted class?
- Compare conceptually to CNN methods like CAM/Grad-CAM. What's the advantage of built-in attention for interpretability?
- Any unusual behavior (attending to background/irrelevant regions)? Are heads specialized?

> TODO: your answer here. (Optional: to check head specialization, visualize each head's attention separately instead of averaging -- see optional cell below.)

In [ ]:
# OPTIONAL: visualize each head separately (instead of averaged) to inspect specialization
@torch.no_grad()
def get_per_head_attention(img, model, processor):
    inputs = processor(images=img, return_tensors="pt").to(device)
    outputs = model(**inputs, output_attentions=True)
    last_layer_attn = outputs.attentions[-1][0]  # (num_heads, 197, 197)
    num_heads = last_layer_attn.shape[0]
    maps = []
    for h in range(num_heads):
        cls_attn = last_layer_attn[h, 0, 1:]
        n = int(np.sqrt(cls_attn.shape[0]))
        maps.append(cls_attn.reshape(n, n).cpu().numpy())
    return maps

sample_img = list(images.values())[0]
head_maps = get_per_head_attention(sample_img, model, processor)
n_heads = len(head_maps)
fig, axes = plt.subplots(1, n_heads, figsize=(2*n_heads, 2))
for h, ax in enumerate(axes):
    ax.imshow(head_maps[h], cmap="viridis")
    ax.set_title(f"head {h}", fontsize=8)
    ax.axis("off")
plt.suptitle("Per-head CLS attention (final layer)")
plt.savefig("../figures/task2_per_head_attention.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Patch-Masking Experiment

In [ ]:
# We need a labeled dataset to measure accuracy under masking, not just single images.
# Use a small slice of a torchvision dataset with ImageNet-compatible labels, or
# simplest: use CIFAR-10/STL-10 test set and just track top-1 label-agreement-with-unmasked
# (since CIFAR/STL classes don't map 1:1 to ImageNet-1k, exact accuracy against ground truth
# isn't meaningful here -- instead we measure how much the prediction changes/degrades
# confidence relative to the unmasked prediction, which is a valid robustness proxy).

from torchvision import datasets, transforms

eval_tf = transforms.Compose([transforms.Resize((224, 224))])
stl_test = datasets.STL10(root="../data", split="test", download=True, transform=eval_tf)

N_EVAL = 100  # keep small for speed
eval_subset = [stl_test[i][0] for i in range(N_EVAL)]

In [ ]:
@torch.no_grad()
def predict_with_patch_mask(img, model, processor, mask_fraction=0.0, mode="random", seed=0):
    """mode: 'random' masks random patches; 'center' masks a contiguous center block."""
    inputs = processor(images=img, return_tensors="pt").to(device)
    pixel_values = inputs["pixel_values"]  # (1, 3, 224, 224)

    patch_size = model.config.patch_size
    n_side = 224 // patch_size  # 14
    n_patches = n_side * n_side
    n_mask = int(mask_fraction * n_patches)

    if n_mask > 0:
        if mode == "random":
            rng = np.random.default_rng(seed)
            mask_idx = rng.choice(n_patches, size=n_mask, replace=False)
        elif mode == "center":
            # pick patches closest to the center
            coords = [(i, j) for i in range(n_side) for j in range(n_side)]
            center = (n_side / 2, n_side / 2)
            coords.sort(key=lambda c: (c[0]-center[0])**2 + (c[1]-center[1])**2)
            mask_idx = [c[0]*n_side + c[1] for c in coords[:n_mask]]
        else:
            raise ValueError(mode)

        pixel_values = pixel_values.clone()
        for idx in mask_idx:
            pi, pj = idx // n_side, idx % n_side
            pixel_values[:, :, pi*patch_size:(pi+1)*patch_size, pj*patch_size:(pj+1)*patch_size] = 0.0

    outputs = model(pixel_values=pixel_values)
    logits = outputs.logits
    pred_idx = logits.argmax(-1).item()
    conf = torch.softmax(logits, dim=-1)[0, pred_idx].item()
    return pred_idx, conf


mask_fractions = [0.0, 0.1, 0.25, 0.4, 0.6, 0.8]
results_random, results_center = [], []

for frac in mask_fractions:
    agree_random, agree_center = 0, 0
    for img in eval_subset:
        base_pred, _ = predict_with_patch_mask(img, model, processor, mask_fraction=0.0)
        pred_r, _ = predict_with_patch_mask(img, model, processor, mask_fraction=frac, mode="random")
        pred_c, _ = predict_with_patch_mask(img, model, processor, mask_fraction=frac, mode="center")
        agree_random += int(pred_r == base_pred)
        agree_center += int(pred_c == base_pred)
    results_random.append(agree_random / len(eval_subset))
    results_center.append(agree_center / len(eval_subset))
    print(f"mask_fraction={frac}: random-mask agreement={results_random[-1]:.3f}, center-mask agreement={results_center[-1]:.3f}")

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(mask_fractions, results_random, marker="o", label="random masking")
plt.plot(mask_fractions, results_center, marker="o", label="center masking")
plt.xlabel("fraction of patches masked")
plt.ylabel("agreement with unmasked prediction")
plt.title("ViT robustness to patch masking")
plt.legend()
plt.savefig("../figures/task2_patch_masking.png", dpi=150, bbox_inches="tight")
plt.show()

**Discussion:** How robust is the ViT to missing patches, and why? Compare random vs. structured (center) masking.

> TODO: your answer here.

## 5. CLS Token vs Mean-Pooled Patch Tokens

In [ ]:
# Use STL-10 (has real labels, 10 classes) to train linear probes on frozen ViT features.
from torchvision import datasets as tv_datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

stl_train = tv_datasets.STL10(root="../data", split="train", download=True, transform=eval_tf)

N_PROBE = 500  # keep small for speed; increase if time allows

@torch.no_grad()
def extract_cls_and_mean_features(dataset, n_samples, model, processor):
    cls_feats, mean_feats, labels = [], [], []
    for i in range(n_samples):
        img, label = dataset[i]
        inputs = processor(images=img, return_tensors="pt").to(device)
        outputs = model.vit(**inputs)  # base ViT, not classification head -- gives hidden states
        last_hidden = outputs.last_hidden_state[0]  # (197, 768)
        cls_feats.append(last_hidden[0].cpu().numpy())        # CLS token
        mean_feats.append(last_hidden[1:].mean(0).cpu().numpy())  # mean of patch tokens
        labels.append(label)
    return np.array(cls_feats), np.array(mean_feats), np.array(labels)

cls_feats, mean_feats, labels = extract_cls_and_mean_features(stl_train, N_PROBE, model, processor)
print(cls_feats.shape, mean_feats.shape, labels.shape)

In [ ]:
X_cls_train, X_cls_test, X_mean_train, X_mean_test, y_train, y_test = train_test_split(
    cls_feats, mean_feats, labels, test_size=0.3, random_state=0, stratify=labels
)

probe_cls = LogisticRegression(max_iter=1000).fit(X_cls_train, y_train)
probe_mean = LogisticRegression(max_iter=1000).fit(X_mean_train, y_train)

acc_cls = probe_cls.score(X_cls_test, y_test)
acc_mean = probe_mean.score(X_mean_test, y_test)

print(f"CLS-token linear probe accuracy: {acc_cls:.4f}")
print(f"Mean-pooled linear probe accuracy: {acc_mean:.4f}")

**Discussion:** Which pooling method performs better, and why? How might pooling choice interact with pretraining objective?

> TODO: your answer here.